In [0]:
%run
./00_config


In [0]:
print("🏗️ Deploying database tables with source and grain documentation...")

# 2. Deploy bronze.customers_raw
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {table_bronze_customers} (
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    date_of_birth STRING,
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _batch_id STRING
  )
  USING delta
  COMMENT "Source: landing_volume/customers/customers.csv | Grain: One row represents a unique human bank customer.";
""")

# 3. Deploy bronze.accounts_raw
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {table_bronze_accounts} (
    account_id STRING,
    customer_id STRING,
    branch_id STRING,
    account_type STRING,
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _batch_id STRING
  )
  USING delta
  COMMENT "Source: landing_volume/accounts/accounts.csv | Grain: One row represents a unique financial account product contract.";
""")

# 4. Deploy bronze.branches_raw
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {table_bronze_branches} (
    branch_id STRING,
    branch_name STRING,
    country STRING,
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _batch_id STRING
  )
  USING delta
  COMMENT "Source: landing_volume/branches/branches.csv | Grain: One row represents a distinct physical retail bank office or branch location.";
""")

# 5. Deploy bronze.transactions_raw
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {table_bronze_transactions} (
    transaction_id STRING,
    account_id STRING,
    transaction_date STRING,
    amount STRING,
    type STRING,
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _batch_id STRING
  )
  USING delta
  COMMENT "Source: landing_volume/transactions/transactions_batch_01.csv | Grain: One row represents an isolated financial ledger event or balance movement.";
""")

print("🥉 All four specialized Bronze Delta tables deployed successfully.")

# 6. Deploy the fifth table: ops.ingestion_audit (Fulfills the new audit table requirement)
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {table_audit_reconcile} (
    batch_id STRING COMMENT 'Unique identifier mapping directly to the _batch_id in raw tables',
    target_table STRING COMMENT 'The fully qualified destination table written to during execution',
    source_path STRING COMMENT 'The directory folder path scanned by the ingestion engine',
    files_loaded LONG COMMENT 'The specific number of new source files processed in this batch run',
    rows_loaded LONG COMMENT 'The total number of data records successfully committed to disk',
    load_ts TIMESTAMP COMMENT 'The precise system clock timestamp tracking when the audit entry was finalized'
  )
  USING delta
  COMMENT "Source: Generated internally by pipeline ingestion notebooks | Grain: One row represents an execution summary of a distinct notebook ingestion batch run.";
""")

print("⚙️ Operational pipeline ledger table 'ops.ingestion_audit' deployed successfully.")


In [0]:
%sql
DESCRIBE TABLE ops.ingestion_audit